[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/learning_methods/05_xgboost/XGBoost.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/learning_methods/05_xgboost/XGBoost.ipynb)

# Notebook 05 — XGBoost
## Learning from Mistakes, Step by Step

**Dataset**: Titanic passengers
**Question we answer**: *Did this passenger survive?*
**Learning type**: Supervised Gradient Boosting — sequential ensemble

---

## Section 1 — What Is XGBoost?

### In plain English

Random Forest asks 100 doctors all at once and takes a vote.
**XGBoost asks them one at a time — each doctor specialises in fixing the previous one's mistakes.**

> **Round 1**: Doctor 1 makes predictions. Some passengers are right, some wrong.
> **Round 2**: Doctor 2 focuses *only* on the passengers Doctor 1 got wrong.
> **Round 3**: Doctor 3 focuses on the passengers Doctor 2 *still* got wrong.
> … repeat 100 times …
> **Final answer**: Sum all 100 doctors' corrections together.

Each "doctor" is a small, shallow decision tree (a **weak learner**).
Each tree learns to fix the **residual errors** left by all previous trees.
This sequential error-correction process is called **boosting**.

### XGBoost vs Random Forest

| | Random Forest | XGBoost |
|---|---|---|
| Trees built | In parallel, independently | Sequentially, each corrects previous |
| Each tree sees | Random bootstrap sample | Residual errors from previous trees |
| Combination | Majority vote | Weighted sum of corrections |
| Handles missing values | No (sklearn) | **Yes — natively** |
| Typical Titanic accuracy | ~82–84% | ~84–86% |

> XGBoost = e**X**treme **G**radient **Boost**ing.

## Section 2 — How Does It Learn?

### Boosting in 4 steps

**Step 1 — Start with a constant prediction**
Initial prediction for everyone: the training survival rate (~38%).

**Step 2 — Compute residuals**
For each passenger: how wrong is the current prediction?
- Passenger survived (1), prediction was 0.38 → residual = +0.62 (we under-predicted)
- Passenger did not survive (0), prediction was 0.38 → residual = −0.38 (we over-predicted)

**Step 3 — Train a small tree to predict the residuals**
The next tree learns: "where is the current model most wrong, and by how much?"

**Step 4 — Update predictions and repeat**
```
new_prediction = old_prediction + (learning_rate × correction_from_tree)
```
After 100+ rounds, the accumulated small corrections produce a highly accurate combined prediction.

> **Why "gradient"?** The residuals are literally the negative gradient of the loss function.
> XGBoost performs gradient descent — but adjusting the *predictions* directly, not weights.

## Section 3 — What Does the Data Need?

### XGBoost's unique strengths

| Requirement | Logistic Reg | Decision Tree | Random Forest | **XGBoost** |
|---|---|---|---|---|
| Feature scaling | Required | Not needed | Not needed | **Not needed** |
| Numeric encoding | Required | Required | Required | Required |
| Missing values | Not allowed | Not allowed | Not allowed | **Handled natively** |
| Distribution shape | Matters | Irrelevant | Irrelevant | **Irrelevant** |
| Extreme outliers | Problematic | Tolerates | Tolerates | **Tolerates** |

### Two approaches to missing values

**Option A — Fill before training** (what we did in earlier notebooks):
```python
df['age'] = df['age'].fillna(df['age'].median())
```

**Option B — Pass NaN directly to XGBoost**:
At each split, XGBoost learns which direction (left or right branch) is better for missing values.
It discovers the best strategy from the data itself — often better than median fill.

In [ ]:
# Install XGBoost if not already available
try:
    import xgboost as xgb
    print(f"XGBoost {xgb.__version__} ready")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    import xgboost as xgb
    print(f"Installed XGBoost {xgb.__version__}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

df_raw = sns.load_dataset('titanic')

FEATURES = ['pclass','sex_enc','age','sibsp','parch','log_fare','embarked_enc','family_size']
TARGET   = 'survived'

# ── Version A: pre-filled (same as other notebooks) ──────────────────────────
df_a = df_raw.copy()
df_a['age']          = df_a['age'].fillna(df_a['age'].median())
df_a['fare']         = df_a['fare'].fillna(df_a['fare'].median())
df_a['embarked']     = df_a['embarked'].fillna(df_a['embarked'].mode()[0])
df_a['sex_enc']      = (df_a['sex'] == 'female').astype(int)
df_a['embarked_enc'] = df_a['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df_a['family_size']  = df_a['sibsp'] + df_a['parch'] + 1
df_a['log_fare']     = np.log1p(df_a['fare'])
df_a = df_a.dropna(subset=['survived'])

# ── Version B: NaN left in age — XGBoost handles it ─────────────────────────
df_b = df_raw.copy()
df_b['fare']         = df_b['fare'].fillna(df_b['fare'].median())
df_b['embarked']     = df_b['embarked'].fillna(df_b['embarked'].mode()[0])
df_b['sex_enc']      = (df_b['sex'] == 'female').astype(int)
df_b['embarked_enc'] = df_b['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df_b['family_size']  = df_b['sibsp'] + df_b['parch'] + 1
df_b['log_fare']     = np.log1p(df_b['fare'])
df_b = df_b.dropna(subset=['survived'])

print(f"Missing values in Version A: {df_a[FEATURES].isnull().sum().sum()}")
print(f"Missing values in Version B: {df_b[FEATURES].isnull().sum().sum()}  (age NaN passed to XGBoost)")

## Section 4 — Train the Model and Read the Rules

In [ ]:
from xgboost import XGBClassifier

def split(df):
    X = df[FEATURES].values
    y = df[TARGET].values.astype(int)
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_tr_a, X_te_a, y_tr_a, y_te_a = split(df_a)
X_tr_b, X_te_b, y_tr_b, y_te_b = split(df_b)

xgb_a = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4,
                       random_state=42, eval_metric='logloss', verbosity=0)
xgb_a.fit(X_tr_a, y_tr_a)
acc_a = accuracy_score(y_te_a, xgb_a.predict(X_te_a))

xgb_b = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4,
                       random_state=42, eval_metric='logloss', verbosity=0)
xgb_b.fit(X_tr_b, y_tr_b)
acc_b = accuracy_score(y_te_b, xgb_b.predict(X_te_b))

print(f"XGBoost — filled NaN (Version A) : {acc_a*100:.1f}%")
print(f"XGBoost — raw NaN   (Version B)  : {acc_b*100:.1f}%  (XGBoost handled missing age itself)")
print()
print("Both approaches work. XGBoost's native handling often matches or beats median fill.")

In [ ]:
importance_df = pd.DataFrame({
    'Feature'    : FEATURES,
    'Importance' : xgb_a.feature_importances_.round(4),
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['Feature'], importance_df['Importance'], color='#AB47BC')
ax.set_title('XGBoost Feature Importances\n(contribution across all 100 boosting rounds)', fontsize=12)
ax.set_xlabel('Importance score', fontsize=11)
plt.tight_layout()
plt.show()

## Section 5 — All Four Methods Side by Side

Now that we have studied all four methods, let us compare them on the same Titanic test set.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Common dataset and split
df_c = df_raw.copy()
df_c['age']          = df_c['age'].fillna(df_c['age'].median())
df_c['fare']         = df_c['fare'].fillna(df_c['fare'].median())
df_c['embarked']     = df_c['embarked'].fillna(df_c['embarked'].mode()[0])
df_c['sex_enc']      = (df_c['sex'] == 'female').astype(int)
df_c['embarked_enc'] = df_c['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df_c['family_size']  = df_c['sibsp'] + df_c['parch'] + 1
df_c['log_fare']     = np.log1p(df_c['fare'])
df_c = df_c.dropna(subset=['survived'])

X_all = df_c[FEATURES].values
y_all = df_c[TARGET].values.astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# Logistic Regression (needs scaling)
sc = StandardScaler()
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(sc.fit_transform(X_tr), y_tr)
acc_lr = accuracy_score(y_te, lr.predict(sc.transform(X_te)))

# Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_tr, y_tr)
acc_dt = accuracy_score(y_te, dt.predict(X_te))

# Random Forest
rf_c = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_c.fit(X_tr, y_tr)
acc_rf = accuracy_score(y_te, rf_c.predict(X_te))

# XGBoost
xgb_c = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4,
                       random_state=42, eval_metric='logloss', verbosity=0)
xgb_c.fit(X_tr, y_tr)
acc_xgb = accuracy_score(y_te, xgb_c.predict(X_te))

results = pd.DataFrame({
    'Method'           : ['Logistic Regression','Decision Tree','Random Forest','XGBoost'],
    'Accuracy'         : [f'{acc_lr*100:.1f}%', f'{acc_dt*100:.1f}%', f'{acc_rf*100:.1f}%', f'{acc_xgb*100:.1f}%'],
    'Needs Scaling'    : ['Yes','No','No','No'],
    'Handles Missing'  : ['No','No','No','Yes (native)'],
    'Rules visible as' : ['Weights','Tree diagram','Feature importances','Feature importances'],
})
print("=== All Four Methods on Titanic ===")
print(results.to_string(index=False))

In [ ]:
methods = ['Logistic\nRegression','Decision\nTree','Random\nForest','XGBoost']
accs    = [acc_lr*100, acc_dt*100, acc_rf*100, acc_xgb*100]
colors  = ['#5C6BC0','#FFA726','#26A69A','#AB47BC']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(methods, accs, color=colors, width=0.5)
ax.set_ylim(65, 95)
ax.set_ylabel('Test accuracy (%)', fontsize=12)
ax.set_title('All Four Methods on Titanic — Survival Prediction Accuracy', fontsize=13)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.4, f'{acc:.1f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Key takeaways:')
print('  1. Gradient methods (Logistic Regression) need feature scaling; tree methods do not')
print('  2. Ensembles (Random Forest, XGBoost) outperform single models')
print('  3. XGBoost uniquely handles missing values without imputation')
print('  4. No single method always wins — the right choice depends on data size,')
print('     interpretability needs, and whether missing values exist')